# Multi-Touch Attribution Model - Interactive Analysis

This notebook demonstrates how to use the Markov Chain attribution model to:
1. Analyze customer journey data across 6 marketing channels
2. Identify undervalued channels
3. Recommend budget reallocation (15% or more)

## Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from markov_attribution import MarkovAttribution
from data_generator import create_scenario_with_undervalued_channel
from visualization import (
    plot_attribution_comparison,
    plot_budget_allocation,
    plot_undervalued_channels,
    create_summary_report
)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries loaded successfully!")

## Step 1: Load Customer Journey Data

We'll use sample data representing customer journeys across 6 marketing channels.

In [ ]:
# Generate sample data
paths_df, current_budget = create_scenario_with_undervalued_channel()

print(f"Dataset contains {len(paths_df)} unique customer journey paths")
print(f"Total conversions: {paths_df['conversions'].sum():,}")
print(f"Total non-conversions: {paths_df['non_conversions'].sum():,}")
print(f"\nCurrent marketing budget: ${sum(current_budget.values()):,}")

# Display sample paths
print("\nSample customer journey paths:")
paths_df.head(10)

## Step 2: Fit the Markov Chain Attribution Model

In [ ]:
# Initialize and fit the model
model = MarkovAttribution()
model.fit(paths_df)

print("✓ Model fitted successfully!")
print(f"\nAnalyzed {len(model.channels)} marketing channels:")
for channel in model.channels:
    print(f"  • {channel}")

## Step 3: View Attribution Results

The attribution weight shows each channel's contribution to conversions.

In [ ]:
# Get attribution results
attribution_df = model.get_attribution_dataframe()
attribution_df

In [ ]:
# Visualize attribution
fig = plot_attribution_comparison(attribution_df, figsize=(14, 6))
plt.show()

## Step 4: Analyze Current Budget Allocation

In [ ]:
# Show current budget
total_budget = sum(current_budget.values())
budget_df = pd.DataFrame([
    {
        'Channel': channel,
        'Budget': f"${budget:,}",
        'Share': f"{budget/total_budget:.1%}",
        'Attribution': f"{model.attributions.get(channel, 0):.1%}",
        'Gap': f"{(model.attributions.get(channel, 0) - budget/total_budget):.1%}"
    }
    for channel, budget in current_budget.items()
])

budget_df.sort_values('Gap', ascending=False)

## Step 5: Identify Undervalued Channels

These channels deserve more budget based on their contribution.

In [ ]:
# Identify undervalued channels
undervalued_df = model.identify_undervalued_channels(current_budget, threshold=0.03)

print(f"Found {len(undervalued_df)} undervalued channels:\n")
undervalued_df

In [ ]:
# Visualize undervalued channels
if len(undervalued_df) > 0:
    fig = plot_undervalued_channels(undervalued_df, figsize=(14, 6))
    plt.show()

## Step 6: Budget Reallocation Recommendations

In [ ]:
# Get budget recommendations
allocation_df = model.recommend_budget_allocation(total_budget, current_budget)
allocation_df

In [ ]:
# Calculate reallocation metrics
increase_df = allocation_df[allocation_df['budget_change'] > 0]
total_reallocation = increase_df['budget_change'].sum()
reallocation_pct = total_reallocation / total_budget * 100

print("BUDGET REALLOCATION SUMMARY")
print("=" * 60)
print(f"Total Budget to Reallocate: ${total_reallocation:,.0f}")
print(f"Reallocation Percentage: {reallocation_pct:.1f}%")
print()

print("Channels Gaining Budget:")
for _, row in increase_df.iterrows():
    print(f"  • {row['channel']:15s}: +${row['budget_change']:8,.0f} ({row['budget_change_pct']:+6.1f}%)")

decrease_df = allocation_df[allocation_df['budget_change'] < 0]
print("\nChannels Losing Budget:")
for _, row in decrease_df.iterrows():
    print(f"  • {row['channel']:15s}: ${row['budget_change']:9,.0f} ({row['budget_change_pct']:+6.1f}%)")

In [ ]:
# Visualize budget allocation
fig = plot_budget_allocation(allocation_df, figsize=(16, 6))
plt.show()

## Step 7: Comprehensive Summary Report

In [ ]:
# Create comprehensive summary
fig = create_summary_report(model, current_budget)
plt.show()

## Key Insights

Run the cell below to see the key takeaways from this analysis.

In [ ]:
print("KEY INSIGHTS")
print("=" * 60)

# Most effective channel
top_channel = attribution_df.iloc[0]
print(f"\n1. Most Effective Channel: {top_channel['channel']}")
print(f"   Attribution Weight: {top_channel['attribution']:.1%}")
print(f"   Removal Effect: {top_channel['removal_effect']:.4f}")

# Most undervalued
if len(undervalued_df) > 0:
    most_undervalued = undervalued_df.iloc[0]
    print(f"\n2. Most Undervalued Channel: {most_undervalued['channel']}")
    print(f"   Current Budget Share: {most_undervalued['current_budget_share']:.1%}")
    print(f"   Should Be: {most_undervalued['attribution_weight']:.1%}")
    print(f"   Undervaluation Gap: {most_undervalued['undervaluation']:.1%}")

# Reallocation summary
print(f"\n3. Recommended Budget Reallocation: {reallocation_pct:.1f}%")
print(f"   Amount: ${total_reallocation:,.0f}")

if reallocation_pct >= 15:
    print(f"\n✓ This analysis identifies channels for {reallocation_pct:.1f}% budget reallocation,")
    print("  exceeding the 15% objective.")

print("\nCONCLUSION")
print("=" * 60)
print("The Markov Chain attribution model successfully identifies undervalued")
print("marketing channels and provides data-driven recommendations for optimal")
print("budget allocation based on actual contribution to conversions.")